# CryptoQuant HDF5 数据查看

这个 notebook 用于分别查看 `data/crypto_quant.h5` 中的各个表。

使用顺序：先运行路径和工具函数单元格，再运行总览单元格或下面对应表的查看单元格。默认只展示前若干行，避免一次性输出整张表。

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import tables

try:
    from IPython.display import display
except ImportError:
    def display(value):
        print(value)

# 兼容从仓库根目录或 data/ 目录启动 Jupyter。
_candidates = [
    Path('data/crypto_quant.h5'),
    Path('crypto_quant.h5'),
    Path.cwd().parent / 'data' / 'crypto_quant.h5',
]
H5_PATH = next((path.resolve() for path in _candidates if path.is_file()), None)
if H5_PATH is None:
    raise FileNotFoundError(
        '找不到 crypto_quant.h5，请确认 notebook 位于 data/ 目录，'
        '或从仓库根目录启动，并先运行 data/update_crypto_quant.py'
    )

print(f'HDF5 文件: {H5_PATH}')
print(f'文件大小: {H5_PATH.stat().st_size / 1024**2:.2f} MB')

HDF5 文件: /Users/dmiwu/work/PythonProject/cryptoFactorAnalyze/data/crypto_quant.h5
文件大小: 183.16 MB


## 工具函数

- `list_tables()`：列出所有表、行数、字段和日期范围。
- `load_table(name)`：加载指定表的完整 DataFrame。
- `preview_table(name)`：按行数和日期范围查看指定表。

表名可以写成 `cmc100_daily` 或 `/cmc100_daily`。

In [2]:
def _table_name(name: str) -> str:
    return name.lstrip('/')


def _stored_columns(storer) -> list[str]:
    """从 HDF5 storer 元数据读取 DataFrame 的真实字段名。"""
    for axis, columns in getattr(storer.attrs, 'non_index_axes', []):
        if axis == 1:
            return list(columns)
    return [column for column in storer.table.colnames if not column.startswith('values_block_')]


def _decode_string(value):
    if isinstance(value, (bytes, np.bytes_)):
        return value.decode('utf-8', errors='replace').rstrip('\x00')
    return value


def _datetime_unit(dtype) -> str:
    text = str(dtype)
    return text.split('[', 1)[1].split(']', 1)[0] if '[' in text else 'ns'


def _read_hdf_table(name: str) -> pd.DataFrame:
    """通过 PyTables 读取，兼容 pandas 1.5 对不同 datetime unit 的断言问题。"""
    name = _table_name(name)
    with pd.HDFStore(H5_PATH, mode='r') as store:
        storer = store.get_storer(name)
        logical_columns = list(storer.attrs.non_index_axes[0][1])
        axes = list(storer.values_axes)
        index_name = next((item[1] for item in getattr(storer.attrs, 'index_cols', []) if item[0] == 0), 'index')

    column_data = {}
    with tables.open_file(H5_PATH, mode='r') as h5:
        node = h5.get_node(f'/{name}/table')
        index_values = node.col('index')
        for axis in axes:
            raw = node.col(axis.name)
            names = list(axis.values)
            if len(names) == 1 and getattr(raw, 'ndim', 1) == 2:
                raw = raw[:, 0]
            if str(axis.kind).startswith('datetime64') or str(axis.dtype).startswith('datetime64['):
                unit = _datetime_unit(axis.dtype)
                if getattr(raw, 'ndim', 1) == 1:
                    converted = pd.to_datetime(raw, unit=unit, errors='coerce', utc=True)
                    if getattr(axis, 'tz', None) is None:
                        converted = converted.tz_localize(None)
                    column_data[names[0]] = converted
                else:
                    for index, column in enumerate(names):
                        converted = pd.to_datetime(raw[:, index], unit=unit, errors='coerce', utc=True)
                        if getattr(axis, 'tz', None) is None:
                            converted = converted.tz_localize(None)
                        column_data[column] = converted
            elif len(names) == 1:
                values = raw
                if axis.kind == 'string':
                    values = [_decode_string(value) for value in values]
                column_data[names[0]] = values
            else:
                for index, column in enumerate(names):
                    column_data[column] = raw[:, index]
    result = pd.DataFrame({column: column_data[column] for column in logical_columns}, index=index_values)
    result.index.name = index_name
    return result


def list_tables() -> pd.DataFrame:
    descriptors = []
    with pd.HDFStore(H5_PATH, mode='r') as store:
        for key in store.keys():
            name = _table_name(key)
            storer = store.get_storer(name)
            descriptors.append((name, storer.nrows, _stored_columns(storer)))
    rows = []
    for name, row_count, columns in descriptors:
        date_range = ''
        date_column = next((column for column in ('date', 'decision_date', 'funding_time') if column in columns), None)
        if date_column is not None:
            dates = pd.to_datetime(_read_hdf_table(name)[date_column], errors='coerce', utc=True).dropna()
            if not dates.empty:
                date_range = f'{dates.min()} ~ {dates.max()}'
        rows.append({'table': name, 'rows': row_count, 'columns': ', '.join(columns), 'date_range': date_range})
    return pd.DataFrame(rows).sort_values('table').reset_index(drop=True)


def load_table(name: str) -> pd.DataFrame:
    """加载指定表的完整内容；大表建议优先使用 preview_table。"""
    return _read_hdf_table(name)


def preview_table(
    name: str,
    rows: int = 10,
    start_date: str | None = None,
    end_date: str | None = None,
) -> pd.DataFrame:
    """查看单张表，可选按 date/decision_date/funding_time 过滤。"""
    frame = load_table(name)
    date_column = next((column for column in ('date', 'decision_date', 'funding_time') if column in frame.columns), None)
    if date_column is not None and (start_date or end_date):
        values = pd.to_datetime(frame[date_column], errors='coerce', utc=True)
        if start_date:
            values_start = pd.Timestamp(start_date, tz='UTC')
            frame = frame.loc[values >= values_start]
            values = values.loc[frame.index]
        if end_date:
            values_end = pd.Timestamp(end_date, tz='UTC') + pd.Timedelta(days=1)
            frame = frame.loc[values < values_end]
    return frame.head(rows).reset_index(drop=True)


def show_table(name: str, rows: int = 10, start_date: str | None = None, end_date: str | None = None) -> None:
    print(f'/{_table_name(name)}')
    display(preview_table(name, rows=rows, start_date=start_date, end_date=end_date))

## 1. 所有表总览

In [3]:
tables_overview = list_tables()
display(tables_overview)

,table,rows,columns,date_range
0,_metadata,329,"key, value, updated_at_utc",
1,cmc100_constituents,91400,"date, cmc_id, symbol, name, weight",2024-01-01 00:00:00+00:00 ~ 2026-08-27 00:00:0...
2,cmc100_daily,914,"date, index_value, source_update_time, fetched...",2024-01-01 00:00:00+00:00 ~ 2026-08-27 00:00:0...
3,funding_events,539415,"funding_time, symbol, funding_rate, mark_price...",2023-07-05 00:00:00+00:00 ~ 2026-08-27 20:00:0...
4,futures_contracts,155,"cmc_id, cmc_symbol, binance_symbol, base_asset...",
5,klines_daily,131244,"date, symbol, open_time, close_time, open, hig...",2023-07-05 00:00:00+00:00 ~ 2026-08-26 00:00:0...
6,research_panel_daily,46850,"date, binance_symbol, open_time, close_time, o...",2024-01-02 00:00:00+00:00 ~ 2026-08-26 00:00:0...
7,universe_monthly,1600,"decision_date, effective_date, effective_end_d...",2024-01-01 00:00:00+00:00 ~ 2026-08-01 00:00:0...


## 2. 分别查看每个表

下面每个单元格只展示前 10 行。可以修改 `rows`，也可以增加 `start_date` / `end_date`，例如：

```python
show_table('research_panel_daily', rows=20, start_date='2024-01-01', end_date='2024-01-10')
```

In [4]:
show_table('cmc100_daily')
show_table('cmc100_constituents')

/cmc100_daily


,date,index_value,source_update_time,fetched_at_utc
0,2024-01-01,100.00,2024-01-01,2026-09-04 05:18:11.483919
1,2024-01-02,104.23,2024-01-02,2026-09-04 05:18:11.483919
2,2024-01-03,104.90,2024-01-03,2026-09-04 05:18:11.483919
3,2024-01-04,99.32,2024-01-04,2026-09-04 05:18:11.483919
4,2024-01-05,102.37,2024-01-05,2026-09-04 05:18:11.483919
5,2024-01-06,101.54,2024-01-06,2026-09-04 05:18:11.483919
6,2024-01-07,100.26,2024-01-07,2026-09-04 05:18:11.483919
7,2024-01-08,99.28,2024-01-08,2026-09-04 05:18:11.483919
8,2024-01-09,105.59,2024-01-09,2026-09-04 05:18:11.483919
9,2024-01-10,104.01,2024-01-10,2026-09-04 05:18:11.483919


/cmc100_constituents


,date,cmc_id,symbol,name,weight
0,2024-01-01,1,BTC,Bitcoin,56.38
1,2024-01-01,2,LTC,Litecoin,0.37
2,2024-01-01,52,XRP,XRP,2.27
3,2024-01-01,74,DOGE,Dogecoin,0.87
4,2024-01-01,328,XMR,Monero,0.21
5,2024-01-01,512,XLM,Stellar,0.25
6,2024-01-01,1027,ETH,Ethereum,18.67
7,2024-01-01,1321,ETC,Ethereum Classic,0.22
8,2024-01-01,1376,NEO,Neo,0.07
9,2024-01-01,1518,MKR,Maker,0.11


In [5]:
show_table('futures_contracts')
show_table('klines_daily')

/futures_contracts


,cmc_id,cmc_symbol,binance_symbol,base_asset,quote_asset,contract_type,onboard_date,status,mapping_source,valid_from,valid_to
0,1,BTC,BTCUSDT,BTC,USDT,PERPETUAL,2019-09-08,TRADING,current_exchange_info,2019-09-08,NaT
1,2,LTC,LTCUSDT,LTC,USDT,PERPETUAL,2020-01-09,TRADING,current_exchange_info,2020-01-09,NaT
2,52,XRP,XRPUSDT,XRP,USDT,PERPETUAL,2020-01-06,TRADING,current_exchange_info,2020-01-06,NaT
3,74,DOGE,DOGEUSDT,DOGE,USDT,PERPETUAL,2020-07-10,TRADING,current_exchange_info,2020-07-10,NaT
4,131,DASH,DASHUSDT,DASH,USDT,PERPETUAL,2020-02-04,TRADING,current_exchange_info,2020-02-04,NaT
5,328,XMR,XMRUSDT,XMR,USDT,PERPETUAL,2020-02-03,TRADING,current_exchange_info,2020-02-03,NaT
6,512,XLM,XLMUSDT,XLM,USDT,PERPETUAL,2020-01-20,TRADING,current_exchange_info,2020-01-20,NaT
7,1027,ETH,ETHUSDT,ETH,USDT,PERPETUAL,2019-11-27,TRADING,current_exchange_info,2019-11-27,NaT
8,1321,ETC,ETCUSDT,ETC,USDT,PERPETUAL,2020-01-16,TRADING,current_exchange_info,2020-01-16,NaT
9,1376,NEO,NEOUSDT,NEO,USDT,PERPETUAL,2020-02-17,TRADING,current_exchange_info,2020-02-17,NaT


/klines_daily


,date,symbol,open_time,close_time,open,high,low,close,volume,quote_volume,trade_count,taker_buy_base_volume,taker_buy_quote_volume
0,2023-07-05,1INCHUSDT,NaT,2023-07-05 23:59:59.999,0.32680,0.33010,0.30920,0.31440,5.310754e+07,NaN,108100,2.556102e+07,8.153069e+06
1,2023-07-05,AAVEUSDT,NaT,2023-07-05 23:59:59.999,77.04000,80.63000,71.67000,74.59000,3.562470e+06,NaN,725039,1.760033e+06,1.343816e+08
2,2023-07-05,ADAUSDT,NaT,2023-07-05 23:59:59.999,0.29220,0.29610,0.27900,0.28370,8.249785e+08,NaN,340787,4.034387e+08,1.157893e+08
3,2023-07-05,ALGOUSDT,NaT,2023-07-05 23:59:59.999,0.12390,0.12520,0.11690,0.11970,3.197384e+08,NaN,118495,1.591068e+08,1.919990e+07
4,2023-07-05,APEUSDT,NaT,2023-07-05 23:59:59.999,2.14800,2.16700,2.03600,2.07700,6.944466e+07,NaN,303392,3.300958e+07,6.921051e+07
5,2023-07-05,APTUSDT,NaT,2023-07-05 23:59:59.999,7.67300,7.95800,7.26400,7.43000,4.336614e+07,NaN,600847,2.110871e+07,1.600930e+08
6,2023-07-05,ARBUSDT,NaT,2023-07-05 23:59:59.999,1.14800,1.15700,1.09080,1.11120,2.188658e+08,NaN,397054,1.042955e+08,1.170746e+08
7,2023-07-05,ARUSDT,NaT,2023-07-05 23:59:59.999,6.00000,6.26600,5.62300,5.82400,7.427887e+06,NaN,233912,3.603622e+06,2.150554e+07
8,2023-07-05,ASTRUSDT,NaT,2023-07-05 23:59:59.999,0.04505,0.04618,0.04337,0.04419,8.966481e+07,NaN,49441,4.317849e+07,1.924674e+06
9,2023-07-05,ATOMUSDT,NaT,2023-07-05 23:59:59.999,9.67800,9.87500,9.24300,9.38800,1.023400e+07,NaN,299308,4.988959e+06,4.760973e+07


In [6]:
show_table('funding_events')
show_table('universe_monthly')

/funding_events


,funding_time,symbol,funding_rate,mark_price,rate_type
0,2023-07-05,1INCHUSDT,0.000100,NaN,Regular
1,2023-07-05,AAVEUSDT,0.000100,NaN,Regular
2,2023-07-05,ADAUSDT,0.000100,NaN,Regular
3,2023-07-05,ALGOUSDT,0.000100,NaN,Regular
4,2023-07-05,APEUSDT,-0.000074,NaN,Regular
5,2023-07-05,APTUSDT,0.000100,NaN,Regular
6,2023-07-05,ARBUSDT,0.000100,NaN,Regular
7,2023-07-05,ARUSDT,-0.000057,NaN,Regular
8,2023-07-05,ASTRUSDT,0.000100,NaN,Regular
9,2023-07-05,ATOMUSDT,0.000076,NaN,Regular


/universe_monthly


,decision_date,effective_date,effective_end_date,cmc_id,cmc_symbol,binance_symbol,market_cap_rank,cmc_weight
0,2024-01-01,2024-01-02,2024-01-31,7278,AAVE,AAVEUSDT,41,0.11
1,2024-01-01,2024-01-02,2024-01-31,2010,ADA,ADAUSDT,6,1.43
2,2024-01-01,2024-01-02,2024-01-31,4030,ALGO,ALGOUSDT,34,0.12
3,2024-01-01,2024-01-02,2024-01-31,21794,APT,APTUSDT,26,0.20
4,2024-01-01,2024-01-02,2024-01-31,11841,ARB,ARBUSDT,32,0.14
5,2024-01-01,2024-01-02,2024-01-31,3794,ATOM,ATOMUSDT,17,0.27
6,2024-01-01,2024-01-02,2024-01-31,5805,AVAX,AVAXUSDT,7,0.96
7,2024-01-01,2024-01-02,2024-01-31,6783,AXS,AXSUSDT,50,0.08
8,2024-01-01,2024-01-02,2024-01-31,1831,BCH,BCHUSDT,15,0.35
9,2024-01-01,2024-01-02,2024-01-31,1839,BNB,BNBUSDT,3,3.23


In [7]:
show_table('research_panel_daily')
show_table('_metadata')

/research_panel_daily


,date,binance_symbol,open_time,close_time,open,high,low,close,volume,quote_volume,...,decision_date,universe_effective_date,market_cap_rank,cmc_weight_at_decision,funding_rate_sum,funding_rate_mean,funding_rate_last,funding_event_count,has_complete_kline,has_complete_funding
0,2024-01-02,AAVEUSDT,NaT,2024-01-02 23:59:59.999,116.3200,120.0700,108.1700,110.4000,8.320606e+05,NaN,...,2024-01-01,2024-01-02,41,NaN,0.001660,0.000553,0.000504,3,True,False
1,2024-01-02,ADAUSDT,NaT,2024-01-02 23:59:59.999,0.6241,0.6380,0.6019,0.6058,5.476557e+08,NaN,...,2024-01-01,2024-01-02,6,NaN,0.001618,0.000539,0.000566,3,True,False
2,2024-01-02,ALGOUSDT,NaT,2024-01-02 23:59:59.999,0.2391,0.2452,0.2243,0.2272,2.759244e+08,NaN,...,2024-01-01,2024-01-02,34,NaN,0.002501,0.000834,0.000828,3,True,False
3,2024-01-02,APTUSDT,NaT,2024-01-02 23:59:59.999,9.9520,10.3740,9.8770,10.2640,1.557144e+07,NaN,...,2024-01-01,2024-01-02,26,NaN,0.001720,0.000573,0.000710,3,True,False
4,2024-01-02,ARBUSDT,NaT,2024-01-02 23:59:59.999,1.7363,1.8433,1.7004,1.7719,4.990985e+08,NaN,...,2024-01-01,2024-01-02,32,NaN,0.002444,0.000815,0.000867,3,True,False
5,2024-01-02,ATOMUSDT,NaT,2024-01-02 23:59:59.999,11.2150,11.4370,10.9310,10.9670,1.400892e+07,NaN,...,2024-01-01,2024-01-02,17,NaN,0.001420,0.000473,0.000441,3,True,False
6,2024-01-02,AVAXUSDT,NaT,2024-01-02 23:59:59.999,41.9790,43.4990,40.3060,40.6540,1.377021e+07,NaN,...,2024-01-01,2024-01-02,7,NaN,0.001829,0.000610,0.000738,3,True,False
7,2024-01-02,AXSUSDT,NaT,2024-01-02 23:59:59.999,9.2620,9.5950,8.8000,8.9490,1.058384e+07,NaN,...,2024-01-01,2024-01-02,50,NaN,0.001313,0.000438,0.000607,3,True,False
8,2024-01-02,BCHUSDT,NaT,2024-01-02 23:59:59.999,268.0800,271.4500,253.7000,257.1900,7.694790e+05,NaN,...,2024-01-01,2024-01-02,15,NaN,0.000740,0.000247,0.000210,3,True,False
9,2024-01-02,BNBUSDT,NaT,2024-01-02 23:59:59.999,313.5300,321.1400,304.9000,311.8800,1.412989e+06,NaN,...,2024-01-01,2024-01-02,3,NaN,-0.000064,-0.000021,0.000000,3,True,False


/_metadata


,key,value,updated_at_utc
0,run_fingerprint,"""3fbad893bfd22d4cd2f137acfa2195b5b701bb0882103...",2026-09-04 05:43:25.233856
1,run_target_as_of,"""2026-08-27T23:59:59+00:00""",2026-09-04 05:43:25.233856
2,mode,"""update""",2026-09-04 05:43:25.233856
3,checkpoint.cmc_through,"""2024-03-18""",2026-09-04 05:43:25.233856
4,mapping_issues,"[{""cmc_id"": 1168, ""cmc_name"": ""Decred"", ""cmc_s...",2026-09-04 05:43:25.233856
5,checkpoint.klines.1INCHUSDT,"{""actual_end"": ""2026-08-26"", ""actual_start"": ""...",2026-09-04 05:43:25.233856
6,checkpoint.funding.1INCHUSDT,"{""actual_end_ms"": 1787846400013, ""actual_start...",2026-09-04 05:43:25.233856
7,checkpoint.klines.2ZUSDT,"{""actual_end"": ""2026-08-26"", ""actual_start"": ""...",2026-09-04 05:43:25.233856
8,checkpoint.funding.2ZUSDT,"{""actual_end_ms"": 1787860800000, ""actual_start...",2026-09-04 05:43:25.233856
9,checkpoint.klines.AAVEUSDT,"{""actual_end"": ""2026-08-26"", ""actual_start"": ""...",2026-09-04 05:43:25.233856


## 3. 常用查询示例

### 查看某天的 Top50

```python
universe = load_table('universe_monthly')
universe[universe['effective_date'].eq(pd.Timestamp('2024-01-02'))]
```

### 查看某个交易对的 K 线

```python
klines = load_table('klines_daily')
klines[klines['symbol'].eq('BTCUSDT')].tail(20)
```

### 查看研究面板的完整性标记

```python
panel = load_table('research_panel_daily')
panel[['date', 'binance_symbol', 'has_complete_kline', 'has_complete_funding']].head()
```

### 查看某天的 Top50

In [ ]:
universe = load_table('universe_monthly')
universe[universe['effective_date'].eq(pd.Timestamp('2024-01-02'))]

,decision_date,effective_date,effective_end_date,cmc_id,cmc_symbol,binance_symbol,market_cap_rank,cmc_weight
index,,,,,,,,
0,2024-01-01,2024-01-02,2024-01-31,7278,AAVE,AAVEUSDT,41,0.11
1,2024-01-01,2024-01-02,2024-01-31,2010,ADA,ADAUSDT,6,1.43
2,2024-01-01,2024-01-02,2024-01-31,4030,ALGO,ALGOUSDT,34,0.12
3,2024-01-01,2024-01-02,2024-01-31,21794,APT,APTUSDT,26,0.20
4,2024-01-01,2024-01-02,2024-01-31,11841,ARB,ARBUSDT,32,0.14
5,2024-01-01,2024-01-02,2024-01-31,3794,ATOM,ATOMUSDT,17,0.27
6,2024-01-01,2024-01-02,2024-01-31,5805,AVAX,AVAXUSDT,7,0.96
7,2024-01-01,2024-01-02,2024-01-31,6783,AXS,AXSUSDT,50,0.08
8,2024-01-01,2024-01-02,2024-01-31,1831,BCH,BCHUSDT,15,0.35
